In [2]:
# 第一部分：文本预处理（在一个句子上完成所有任务）
# 在 Colab 中运行前，先安装必要库
!pip install pymorphy3 natasha razdel

In [6]:
import re
from razdel import tokenize
import pymorphy3
from natasha import (
    Segmenter,
    MorphVocab,
    NewsEmbedding,
    NewsMorphTagger,
    NewsSyntaxParser,
    NewsNERTagger,
    Doc
)

# 示例句子
text = "Мария купила красивую книгу в Москве вчера."

print("=" * 50)
print("原始文本:", text)
print("=" * 50)

# 初始化组件
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
syntax_parser = NewsSyntaxParser(emb)
ner_tagger = NewsNERTagger(emb)

# 创建Doc对象
doc = Doc(text)

# 1. 分词 - 通过segmenter
print("\n1. 分词结果:")
doc.segment(segmenter)
tokens = [token.text for token in doc.tokens]
print(tokens)

# 2. 词性标注
print("\n2. 词性标注结果:")
doc.tag_morph(morph_tagger)
for token in doc.tokens:
    pos = token.pos if hasattr(token, 'pos') and token.pos else "未知"
    print(f"{token.text}: {pos}")

# 3. 词形归并
print("\n3. 词形归并结果:")
for token in doc.tokens:
    token.lemmatize(morph_vocab)
    lemma = token.lemma if hasattr(token, 'lemma') and token.lemma else token.text
    print(f"{token.text} -> {lemma}")

# 4. 命名实体识别
print("\n4. 命名实体识别结果:")
doc.tag_ner(ner_tagger)
for span in doc.spans:
    span.normalize(morph_vocab)
    print(f"实体: {span.text} -> 类型: {span.type}")

# 5. 句法分析 - 修复类型错误
print("\n5. 句法分析结果:")
doc.parse_syntax(syntax_parser)

for token in doc.tokens:
    if hasattr(token, 'rel') and token.rel:
        # 处理 head_id 可能是字符串的情况
        if hasattr(token, 'head_id') and token.head_id:
            try:
                # 尝试转换为整数
                head_idx = int(token.head_id) if str(token.head_id).isdigit() else 0
                if head_idx < len(doc.tokens):
                    head_text = doc.tokens[head_idx].text
                else:
                    head_text = "root"
            except (ValueError, TypeError):
                head_text = "root"
        else:
            head_text = "root"
        print(f"单词: '{token.text}', 词性: {token.pos}, 依存关系: {token.rel}, 父节点: {head_text}")
    else:
        print(f"单词: '{token.text}', 词性: {token.pos if hasattr(token, 'pos') else '?'}")

# 简单显示句法关系
print("\n句法关系:")
for token in doc.tokens:
    if hasattr(token, 'rel') and token.rel and token.rel != 'punct':
        print(f"  {token.text} --> {token.rel}")

原始文本: Мария купила красивую книгу в Москве вчера.

1. 分词结果:
['Мария', 'купила', 'красивую', 'книгу', 'в', 'Москве', 'вчера', '.']

2. 词性标注结果:
Мария: PROPN
купила: VERB
красивую: ADJ
книгу: NOUN
в: ADP
Москве: PROPN
вчера: ADV
.: PUNCT

3. 词形归并结果:
Мария -> мария
купила -> купить
красивую -> красивый
книгу -> книга
в -> в
Москве -> москва
вчера -> вчера
. -> .

4. 命名实体识别结果:
实体: Мария -> 类型: PER
实体: Москве -> 类型: LOC

5. 句法分析结果:
单词: 'Мария', 词性: PROPN, 依存关系: nsubj, 父节点: Мария
单词: 'купила', 词性: VERB, 依存关系: root, 父节点: Мария
单词: 'красивую', 词性: ADJ, 依存关系: amod, 父节点: Мария
单词: 'книгу', 词性: NOUN, 依存关系: obj, 父节点: Мария
单词: 'в', 词性: ADP, 依存关系: case, 父节点: Мария
单词: 'Москве', 词性: PROPN, 依存关系: obl, 父节点: Мария
单词: 'вчера', 词性: ADV, 依存关系: advmod, 父节点: Мария
单词: '.', 词性: PUNCT, 依存关系: punct, 父节点: Мария

句法关系:
  Мария --> nsubj
  купила --> root
  красивую --> amod
  книгу --> obj
  в --> case
  Москве --> obl
  вчера --> advmod


In [9]:
# 第二部分：文本分类 - 两种方法对比
# 使用 sklearn 自带的新闻数据集，简单二分类（选两个类别）

# 安装必要库
!pip install --upgrade gensim nltk scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 18.7 MB/s eta 0:00:00
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [1]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

# 加载数据（只取两个类别，简化二分类任务）
categories = ['rec.sport.baseball', 'sci.space']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

X = newsgroups.data
y = newsgroups.target

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"训练样本数: {len(X_train)}, 测试样本数: {len(X_test)}")
print(f"类别: {newsgroups.target_names}")
print("=" * 60)

# ========== 方法1: TfidfVectorizer ==========
print("\n【方法1】基于 TfidfVectorizer + 逻辑回归")

# 向量化
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 训练分类器
clf1 = LogisticRegression(max_iter=1000, random_state=42)
clf1.fit(X_train_tfidf, y_train)

# 预测
y_pred1 = clf1.predict(X_test_tfidf)

# 评估
acc1 = accuracy_score(y_test, y_pred1)
f11 = f1_score(y_test, y_pred1)
print(f"准确率 (Accuracy): {acc1:.4f}")
print(f"F1分数 (F1-score): {f11:.4f}")

# ========== 方法2: Word2Vec ==========
print("\n【方法2】基于 Word2Vec + 平均词向量 + 逻辑回归")

from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
from gensim.models import Word2Vec
import re

# 预处理: 分词
def tokenize_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    # 去掉标点和数字
    tokens = [t for t in tokens if t.isalpha()]
    return tokens

# 对所有文本分词
tokenized_docs = [tokenize_text(doc) for doc in X_train]

# 训练 Word2Vec 模型
w2v_model = Word2Vec(sentences=tokenized_docs, vector_size=100, window=5, min_count=2, workers=4, seed=42)

# 获取文档向量 (平均词向量)
def get_doc_vector(tokens, model, vector_size=100):
    word_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(word_vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(word_vectors, axis=0)

# 转换成向量
X_train_w2v = np.array([get_doc_vector(doc, w2v_model) for doc in tokenized_docs])

# 测试集处理
tokenized_test = [tokenize_text(doc) for doc in X_test]
X_test_w2v = np.array([get_doc_vector(doc, w2v_model) for doc in tokenized_test])

# 训练分类器
clf2 = LogisticRegression(max_iter=1000, random_state=42)
clf2.fit(X_train_w2v, y_train)

# 预测
y_pred2 = clf2.predict(X_test_w2v)

# 评估
acc2 = accuracy_score(y_test, y_pred2)
f12 = f1_score(y_test, y_pred2)
print(f"准确率 (Accuracy): {acc2:.4f}")
print(f"F1分数 (F1-score): {f12:.4f}")

# ========== 比较结果 ==========
print("\n" + "=" * 60)
print("【模型质量对比】")
print("=" * 60)
print(f"方法1 (TfidfVectorizer):  Accuracy = {acc1:.4f}, F1 = {f11:.4f}")
print(f"方法2 (Word2Vec):         Accuracy = {acc2:.4f}, F1 = {f12:.4f}")
print(f"差异: Accuracy 相差 {abs(acc1-acc2):.4f}")

训练样本数: 1584, 测试样本数: 397
类别: ['rec.sport.baseball', 'sci.space']

【方法1】基于 TfidfVectorizer + 逻辑回归
准确率 (Accuracy): 0.9924
F1分数 (F1-score): 0.9928

【方法2】基于 Word2Vec + 平均词向量 + 逻辑回归


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


准确率 (Accuracy): 0.8690
F1分数 (F1-score): 0.8744

【模型质量对比】
方法1 (TfidfVectorizer):  Accuracy = 0.9924, F1 = 0.9928
方法2 (Word2Vec):         Accuracy = 0.8690, F1 = 0.8744
差异: Accuracy 相差 0.1234
